In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text

In [3]:
SERVER = 'DESKTOP-1478HIH\\SQLEXPRESS'
DATABASE = 'oulad_warehouse'
USERNAME = 'BI_User'
PASSWORD = '123456'

connection_url = f"mssql+pyodbc://{USERNAME}:{PASSWORD}@{SERVER}/{DATABASE}?driver=ODBC+Driver+17+for+SQL+Server"

engine = create_engine(connection_url)

try:
    with engine.connect() as connection:
        dim_modules     = pd.read_sql_query(text("SELECT * FROM dbo.Dim_Modules_Catalog"), connection)
        dim_students    = pd.read_sql_query(text("SELECT * FROM dbo.Dim_Students_Profile"), connection)
        dim_assessments = pd.read_sql_query(text("SELECT * FROM dbo.Dim_Assessments_Info"), connection)
        dim_vle         = pd.read_sql_query(text("SELECT * FROM dbo.Dim_Vle_Resources"), connection)
        fact_enrollment  = pd.read_sql_query(text("SELECT * FROM dbo.Fact_Enrollment"), connection)
        fact_scores      = pd.read_sql_query(text("SELECT * FROM dbo.Fact_Student_Scores"), connection)
        fact_clicks      = pd.read_sql_query(text("SELECT * FROM dbo.Fact_Student_Vle_Clicks"), connection)
    print('All Tables has loaded successfully')

except Exception as e:
    print('Something gone wrong')
    print(e)

All Tables has loaded successfully


## Dashboard 1

In [5]:
resource_summary = fact_clicks.groupby('activity_type')['sum_click'].sum().reset_index()
resource_summary = resource_summary.sort_values(by='sum_click', ascending=False)

print("Analysis 1: VLE Resource Popularity Report:")
print(resource_summary)
print("\n" + "="*50 + "\n")


weekly_trend = fact_clicks.groupby('date')['sum_click'].sum().reset_index()

print("📈 Analysis 2: Weekly VLE Clicks Trend (Top 5 Rows):")
print(weekly_trend.head())
print("\n" + "="*50 + "\n")


active_keys_in_vle = fact_clicks['Student_Key'].unique()
lurkers_df = fact_enrollment[~fact_enrollment['Student_Key'].isin(active_keys_in_vle)]

print(f"Analysis 3: Total 'Lurker' Students Found: {len(lurkers_df)}")
print(lurkers_df[['Student_Key', 'Module_Key', 'final_result']].head())
print("\n" + "="*50 + "\n")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(ax=axes[0], data=resource_summary.head(10), x='sum_click', y='activity_type', palette='Blues_r')
axes[0].set_title('Instructor View: Most Used VLE Resources')
axes[0].set_xlabel('Total Clicks Volume')
axes[0].set_ylabel('Resource Type')

sns.lineplot(ax=axes[1], data=weekly_trend, x='date', y='sum_click', color='orange', linewidth=2)
axes[1].set_title('Academic Trend: Student Clicks Over Time (Weeks)')
axes[1].set_xlabel('Timeline (Days relative to start)')
axes[1].set_ylabel('Total Interactions')
axes[1].grid(True, linestyle='--')


plt.tight_layout()
plt.show()

KeyError: 'activity_type'

## Dashboard 2

In [ ]:
assessment_variance = fact_scores.groupby('assessment_type')['score'].mean().reset_index()

print("📝 Analysis 4: Grading Type Average Scores:")
print(assessment_variance)
print("\n" + "="*50 + "\n")

ate_submission_gap = fact_scores.groupby('is_late_submission')['score'].mean().reset_index()

late_submission_gap['Submission_Status'] = late_submission_gap['is_late_submission'].map({0: 'On-Time', 1: 'Late'})

print("⏳ Analysis 5: Late Submission Impact on Scores:")
print(late_submission_gap[['Submission_Status', 'score']])
print("\n" + "="*50 + "\n")


fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.boxplot(ax=axes[0], data=fact_scores[fact_scores['assessment_type'].isin(['TMA', 'CMA'])], 
            x='assessment_type', y='score', palette='Pastel1')
axes[0].set_title('Instructor Audit: TMA (Manual) vs CMA (Automated) Distribution')
axes[0].set_xlabel('Assessment Type')
axes[0].set_ylabel('Student Scores')

exam_scores = fact_scores[fact_scores['assessment_type'] == 'Exam']
sns.histplot(ax=axes[1], data=exam_scores, x='score', bins=20, kde=True, color='teal')
axes[1].set_title('Classroom Diagnostics: Final Exam Grade Distribution (Bell Curve)')
axes[1].set_xlabel('Final Exam Score')
axes[1].set_ylabel('Number of Students')

plt.tight_layout()
plt.show()

## Dashboard 3

In [ ]:
active_keys_in_vle = fact_clicks['Student_Key'].unique()
vle_inactive_students = fact_enrollment[~fact_enrollment['Student_Key'].isin(active_keys_in_vle)]

student_avg_scores = fact_scores.groupby('Student_Key')['score'].mean().reset_index()
at_risk_scores = student_avg_scores[student_avg_scores['score'] < 40]

at_risk_profiles = dim_students[dim_students['num_of_prev_attempts'] >= 2]
risk_by_attempts = pd.merge(at_risk_profiles, student_avg_scores, on='Student_Key', how='inner')

print("🚨 Analysis 7 & 8: Urgent Classroom Risk Radar:")
print(f"- Students with ZERO VLE Engagement: {len(vle_inactive_students)}")
print(f"- Students with Critical Average Score (< 40): {len(at_risk_scores)}")
print("\n" + "="*50 + "\n")

print("🔍 Analysis 9: Repeated Attempts vs Current Average Score (Top 5):")
print(risk_by_attempts[['Student_Key', 'num_of_prev_attempts', 'score']].head())
print("\n" + "="*50 + "\n")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.scatterplot(ax=axes[0], data=fact_scores, x='date_submitted', y='score', hue='assessment_type', alpha=0.5)
axes[0].set_title('Risk Radar: Submission Day vs Score Achieved')
axes[0].set_xlabel('Day of Submission')
axes[0].set_ylabel('Score')

sns.barplot(ax=axes[1], data=risk_by_attempts, x='num_of_prev_attempts', y='score', palette='Oranges_r')
axes[1].set_title('Academic Burden: Impact of Previous Attempts on Current Scores')
axes[1].set_xlabel('Number of Previous Attempts')
axes[1].set_ylabel('Current Avg Score')

plt.tight_layout()
plt.show()